# Overall efficiency


> To compare efficiency across methods and problems we used the overall efficiency measure introduced in {Villaverde2019}. For optimiser $i$ on benchmark problem $j$, we determined the overall computation time $T_i^{(j)}$ and the number of successful starts $S_i^{(j)}$, yielding the average computation time that the $i$-th optimiser required to produce a successful start as
\begin{equation}
\langle t_{\mathrm{succ},i}^{(j)} \rangle = \frac{T_i^{(j)}}{S_i^{(j)}}.
\end{equation}
The overall efficiency on problem $j$ is then
\begin{equation}
\mathrm{OE}_i^{(j)} = \frac{\min_{i'} \langle t_{\mathrm{succ},i'}^{(j)} \rangle}{\langle t_{\mathrm{succ},i}^{(j)} \rangle}
\in [0,1].
\end{equation}
The best optimiser for a given problem has $\mathrm{OE}=1$; $1/\mathrm{OE}_i^{(j)}$ indicates how many times longer optimiser $i$ must be run, relative to the best method, to obtain one successful start.

To combine the 10 runs, we can either compute OEs per run and average them, or we can combine all runs and compute a single OE.
We do the latter.


## Load data and compute overall efficiency

In [ ]:
import json
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.cm import ScalarMappable
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.lines import Line2D
from PyComplexHeatmap import ClusterMapPlotter, HeatmapAnnotation, anno_barplot

from lib import (
    N_OPTIMIZERS,
    N_PROBLEMS,
    N_RUNS,
    OPTIMIZER_OVERVIEW_PATH,
    PROBLEM_OVERVIEW_PATH,
    RELATIVE_WALLTIME_LIMIT,
    get_threshold,
)

optimizer_df = pd.read_csv(OPTIMIZER_OVERVIEW_PATH)
problem_df = pd.read_csv(PROBLEM_OVERVIEW_PATH)

with open("../data/successful_starts.json") as f:
    successful_starts = json.load(f)

df = pd.DataFrame(successful_starts)

print("Significance level:", df.significance.unique())
assert df.significance.nunique() == 1

df = df.join(
    optimizer_df.set_index(["output_dir"]),
    on="optimizer",
    how="left",
    validate="many_to_one",
)
df = df.join(
    problem_df.set_index(["short"]),
    on="problem",
    how="left",
    validate="many_to_one",
)
assert (df.overall_time_s <= df.walltime_s * RELATIVE_WALLTIME_LIMIT).all(), (
    "Wall time limit violation"
)
df = df.query("excluded == False").query("~is_pysacess")
assert len(df) == N_OPTIMIZERS * N_PROBLEMS * N_RUNS, (
    N_OPTIMIZERS * N_PROBLEMS * N_RUNS,
    len(df),
)

if False:
    # cursory check for overall compute time distribution
    ax = sns.kdeplot(df.overall_time_s, cut=0)
    sns.histplot(
        df.overall_time_s,
        bins=20,
        ax=ax,
        color="C0",
        alpha=0.5,
        stat="density",
    )
    for x in problem_df.walltime_s.unique():
        ax.axvline(
            RELATIVE_WALLTIME_LIMIT * x,
            color="k",
            linestyle=":",
            label="Walltime limit",
        )
    plt.show()

df_by_run = df.copy()

# combine all runs for each problem x optimizer
df = (
    df.groupby(["problem", "optimizer"])
    .agg(
        n_success=pd.NamedAgg(column="n_success", aggfunc="sum"),
        overall_time_s=pd.NamedAgg(column="overall_time_s", aggfunc="sum"),
    )
    .reset_index()
)
assert len(df) == N_OPTIMIZERS * N_PROBLEMS

# average time per successful starts
df["avg_t_succ"] = df.overall_time_s / df.n_success
# (overall_time_s is the wall time. we skip multiplication with the number of cores
#  which would cancel out in the overall_efficiency anyway.)

# minimum avg_t_succ per problem across all optimizers
df["min_avg_t_succ"] = df.groupby(["problem"])["avg_t_succ"].transform("min")

# overall efficiency
df["overall_efficiency"] = df.min_avg_t_succ / df.avg_t_succ
df.fillna({"overall_efficiency": 0.0}, inplace=True)
assert df.overall_efficiency.min() >= 0
assert df.overall_efficiency.max() <= 1

df = df.join(
    problem_df.set_index(["short"]),
    on="problem",
    how="left",
    validate="many_to_one",
)
df = df.join(
    optimizer_df.set_index(["output_dir"]),
    on="optimizer",
    how="left",
    validate="many_to_one",
)

df

## Fig 6B

In [ ]:
df_best_fvals = pd.read_csv("out/best_fvals.csv", index_col="problem")


def get_endpoints(result_file: str | Path, walltime_limit_s: float):
    # We need the smallest fval of each start before the wall time limit
    end_points = []
    with h5py.File(result_file, "r") as f:
        # timestamp of start of *optimization*
        group_to_start_ts = {
            group_name: f[group_name]["start_time"][()] for group_name in f
        }
        # start of earliest *optimization* marks the start of the *run*
        earliest_start_ts = min(group_to_start_ts.values())

        cutoff_ts = earliest_start_ts + walltime_limit_s
        for group_name in f:
            start_ts = group_to_start_ts[group_name]
            if start_ts > cutoff_ts:
                continue
            group = f[group_name]
            mask = start_ts + group["time"] <= cutoff_ts
            best_fval = group["fval"][mask].min(initial=np.inf)
            if np.isfinite(best_fval):
                end_points.append(best_fval)
    return np.array(end_points)

In [ ]:
selected_examples = (
    ("Laske", "fides"),
    ("Laske", "cmaes"),
    ("Laske", "L-BFGS-B"),
)

individual_starts_path = "../data/individual_starts_for_fig_6b.csv"
if not Path(individual_starts_path).exists():
    individual_starts_df = None
    for problem_id, optimizer in selected_examples:
        for i_run in range(1, N_RUNS + 1):
            result_file = Path(
                f"data_samples/{optimizer}_{problem_id}/run_{i_run:02d}.h5"
            )
            assert result_file.exists(), result_file
            end_points = get_endpoints(
                result_file,
                walltime_limit_s=RELATIVE_WALLTIME_LIMIT
                * float(
                    problem_df.query(
                        "short == @problem_id"
                    ).walltime_s.values.squeeze()
                ),
            )
            optimality_gaps = (
                end_points
                - df_best_fvals.loc[problem_id, "marvin_without_pyscat"]
            )
            print(len(optimality_gaps))
            assert np.all(optimality_gaps >= 0), optimality_gaps
            tmp_df = pd.DataFrame(
                {
                    "problem": problem_id,
                    "optimizer": optimizer,
                    "run_idx": i_run,
                    "optimality_gap": optimality_gaps,
                }
            )
            if individual_starts_df is None:
                individual_starts_df = tmp_df
            else:
                individual_starts_df = pd.concat(
                    [individual_starts_df, tmp_df], ignore_index=True
                )
            del tmp_df

    individual_starts_df.to_csv(individual_starts_path, index=False)
else:
    individual_starts_df = pd.read_csv(individual_starts_path)

In [ ]:
def success_over_og(
    optimality_gaps: np.ndarray,
    ax: plt.Axes | None = None,
    og_max=1e4,
    alpha=0.05,
    threshold_color="gray",
):
    optimality_gaps = optimality_gaps[optimality_gaps <= og_max]
    x = np.sort(np.unique(optimality_gaps))
    if x[0] != 0:
        x = np.insert(x, 0, 0)
    if x[-1] != og_max:
        x = np.append(x, og_max)
    y = [(optimality_gaps <= og).sum() for og in x]

    if ax is None:
        fig, ax = plt.subplots(figsize=(3, 2.5))

    ax.step(x, y, where="post")
    ax.set_xscale("symlog", linthresh=1)
    ax.set_yscale("symlog", linthresh=1)
    ax.set_xlim(og_max, 0)
    ax.set_xlabel("Optimality gap threshold")
    ax.set_ylabel("# Successful starts")
    solved_tresh = get_threshold(percentile=1 - alpha, df=1)
    ax.axvline(solved_tresh, color=threshold_color, linestyle=":", zorder=0)
    n_solved_at_threshold = (optimality_gaps <= solved_tresh).sum()
    ax.annotate(
        str(n_solved_at_threshold),
        xy=(solved_tresh, n_solved_at_threshold),
        xytext=(10, 10),
        textcoords="offset points",
        va="bottom",
        ha="left",
        arrowprops=dict(
            arrowstyle="->",
            linewidth=1,
            shrinkA=0,
            shrinkB=0,
            connectionstyle="arc3,rad=0",
        ),
    )
    return ax

In [ ]:
with plt.rc_context(
    rc={
        "figure.figsize": (2, 5),
        "figure.dpi": 300,
        "font.size": 6,
        "lines.markersize": 3,
    }
):
    fig, axs = plt.subplots(3, 1, layout="constrained", sharex=True)
    for (problem, optimizer), ax in zip(selected_examples, axs, strict=False):
        # combined number of starts from all runs
        df_tmp = individual_starts_df.query(
            f"problem == '{problem}' and optimizer == '{optimizer}'"
        )
        ax = success_over_og(
            df_tmp.optimality_gap.values,
            ax=ax,
        )
        ax.set_title(
            f"{optimizer_df.set_index('output_dir').loc[optimizer, 'optimizer_label']} - {problem}"
        )

    plt.savefig("out/Figure6B.svg")
    plt.show()

## Fig 6C

In [ ]:
import matplotlib.patches as patches

df_avg_oe = df
hm_data = df_avg_oe.pivot_table(
    index="optimizer_label", columns="problem", values="overall_efficiency"
)

# The maximum OE per problem must either be 1 or 0.
#  0: We determine the optimality gap from the best value across both sites
#  it's possible we don't get a single successful start on the current site.
assert (hm_data.max().isin((0, 1))).all(), hm_data.max()

cmap = plt.colormaps.get_cmap("Greens").copy()
cmap.set_bad("red")
cell_highlight_color = "#FF00FF"

with plt.rc_context(
    {
        "font.size": 10,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "axes.labelsize": 10,
    }
):
    plt.figure(figsize=(8, 8))

    # right annotation --- average OE per optimizer
    df_row_bar = df_avg_oe.groupby("optimizer_label")[
        "overall_efficiency"
    ].mean()
    row_ha = HeatmapAnnotation(
        test=anno_barplot(
            df_row_bar,
            height=15,
            colors="#008080",
            label="Avg. OE",
            legend=False,
            ylim=(0, 1),
        ),
        axis=0,
        # label_side="bottom",
        label_kws={
            "rotation": 0,
            "fontsize": 8,
            "horizontalalignment": "center",
            "verticalalignment": "bottom",
        },
    )

    # top annotation --- averoge OE per problem
    df_col_bar = df_avg_oe.groupby("problem")["overall_efficiency"].mean()
    col_ha = HeatmapAnnotation(
        bla=anno_barplot(
            df_col_bar,
            height=15,
            colors="#008080",
            label="Avg. OE",
            legend=False,
            ylim=(0, 1),
        ),
        axis=1,
        label_kws={
            "rotation": 90,
            "fontsize": 8,
            "horizontalalignment": "center",
        },
    )

    cm = ClusterMapPlotter(
        data=hm_data
        # order by number of solved
        .loc[
            df_row_bar.sort_values(ascending=False).index,
            df_col_bar.sort_values(ascending=False).index,
        ],
        top_annotation=col_ha,
        right_annotation=row_ha,
        col_split=problem_df.set_index(["short"])
        .loc[df_col_bar.sort_values(ascending=False).index]
        .difficulty,
        col_split_gap=2,
        row_dendrogram=False,
        col_dendrogram=False,
        row_cluster=False,
        col_cluster=False,
        show_rownames=True,
        show_colnames=True,
        row_names_side="left",
        cmap=cmap,
        xticklabels_kws=dict(labelrotation=90),
        # color bar
        legend_kws=dict(
            extend="neither",
        ),
        vmin=0,
        vmax=1,
        label="Overall efficiency",
        linecolor="white",
        linewidth=0.5,
        xlabel="Problem",
        ylabel="Optimisation method",
    )
cm.ax_heatmap.set_aspect("equal")

col_ha.axes.flatten()[0].yaxis.set_label_text(
    col_ha.axes.flatten()[1].yaxis.get_label_text()
)
col_ha.axes.flatten()[1].yaxis.set_label_text("")
col_ha.axes.flatten()[0].yaxis.label.set_visible(True)
col_ha.axes.flatten()[0].yaxis.label.set_fontsize(
    col_ha.axes.flatten()[1].yaxis.label.get_fontsize()
)

row_ha.axes.flatten()[0].xaxis.set_ticks_position("top")
row_ha.axes.flatten()[0].xaxis.set_tick_params(labelrotation=0)

# highlight selected examples
if selected_examples:
    # the heatmap cells are in axes[6], not in ax_heatmap ...
    ax_real = cm.ax_heatmap.figure.axes[6]

    row_labels = cm.row_order[0]
    col_labels = cm.col_order[0]
    for problem, optimizer in selected_examples:
        row_idx = row_labels.index(
            optimizer_df.set_index("output_dir").loc[
                optimizer, "optimizer_label"
            ]
        )
        col_idx = col_labels.index(problem)

        rect = patches.Rectangle(
            (col_idx, row_idx),
            width=1,
            height=1,
            zorder=999,
            edgecolor=cell_highlight_color,
            facecolor="none",
            clip_on=False,
        )
        ax_real.add_patch(rect)

plt.savefig("out/Figure6C.svg", bbox_inches="tight")

In [ ]:
# color bar with histogram for heatmap


def my_hist(
    data: np.ndarray,
    upper_bound=None,
    base_cmap=plt.colormaps.get_cmap("Greens_r"),
    overflow_color=np.array([[1, 0, 0, 1]]),
    edgecolor=None,
    ax: plt.Axes | None = None,
):
    normal_edges = np.linspace(
        data.min(), upper_bound if upper_bound is not None else data.max(), 30
    )
    bin_width = normal_edges[1] - normal_edges[0]

    if upper_bound is not None:
        overflow_max = upper_bound + bin_width

        plot_data = np.minimum(data, overflow_max - 1e-9)
        bins = np.r_[normal_edges, overflow_max]
    else:
        plot_data = data
        bins = normal_edges

    if ax is None:
        _, ax = plt.subplots(figsize=(3, 2))

    counts, edges, patches = ax.hist(plot_data, bins=bins, edgecolor=edgecolor)

    if upper_bound is None:
        colors = base_cmap(np.linspace(0, 1, len(patches)))
    else:
        normal_colors = base_cmap(np.linspace(0, 1, len(patches) - 1))
        colors = np.vstack([normal_colors, overflow_color])

    for patch, color in zip(patches, colors, strict=True):
        patch.set_facecolor(color)

    cmap = ListedColormap(colors)
    norm = BoundaryNorm(edges, cmap.N)

    sm = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    cbar = ax.figure.colorbar(
        sm,
        ax=ax,
        orientation="horizontal",
        pad=0.01,
        boundaries=edges,
        spacing="proportional",
    )

    # Regular continuous ticks
    regular_ticks = np.arange(
        data.min(),
        upper_bound if upper_bound is not None else data.max() + np.spacing(1),
        step=0.5,
    )

    if upper_bound is not None:
        # Add one extra tick centered in overflow bin
        overflow_tick = upper_bound + bin_width / 2

        cbar.set_ticks(np.r_[regular_ticks, overflow_tick])
        cbar.set_ticklabels(
            [str(t) for t in regular_ticks] + [f">{upper_bound:.2f}"]
        )
    else:
        cbar.set_ticks(regular_ticks)
        cbar.set_ticklabels([str(t) for t in regular_ticks])
    cbar.minorticks_off()

    ax.set_xlim(edges[0], edges[-1])
    cbar.ax.set_xlim(edges[0], edges[-1])
    ax.xaxis.set_visible(False)

    cbar.ax.set_xlabel("OE")
    ax.yaxis.set_visible(False)

    cbar.ax.tick_params(which="minor", length=3)
    cbar.ax.tick_params(which="major", length=6)

    for spine in ax.spines.values():
        spine.set_visible(False)


with plt.rc_context(
    {
        "figure.figsize": (0.5, 0.5),
        "figure.dpi": 300,
        "font.size": 10,
        "lines.markersize": 1,
    }
):
    fig, ax = plt.subplots(figsize=(1.5, 1))
    my_hist(
        data=hm_data.values.flatten(),
        edgecolor="black",
        base_cmap=plt.colormaps.get_cmap("Greens"),
        ax=ax,
    )

plt.savefig("out/Figure6C_cbar.svg", bbox_inches="tight")
plt.show()

In [ ]:
hm_data.gt(0.8).sum().sort_values(ascending=False)

In [ ]:
hm_data.gt(0.8).sum(axis=1).sort_values(ascending=False)

## Figs 6 D,E

In [ ]:
# Fig 6D, E -- boxplot with number of success starts (summed across runs) per problem and per optimizer
# stripplot instead of boxplot


def add_split_spines(ax, gap_idx, xmax=None):
    lw = ax.spines["left"].get_linewidth()
    ec = ax.spines["left"].get_edgecolor()

    for spine in ax.spines.values():
        spine.set_visible(False)

    ymin, ymax = ax.get_ylim()
    xmin, xmax_ = ax.get_xlim()
    if xmax is None:
        xmax = xmax_

    # categories are at integer x; gap at gap_idx means left group ends at
    # gap_idx - 0.5 and right group starts at gap_idx + 0.5
    x_split_l = gap_idx - 0.5
    x_split_r = gap_idx + 0.5

    kw = dict(
        transform=ax.transData,
        color=ec,
        linewidth=lw,
        clip_on=False,
        solid_capstyle="butt",
    )

    for x0, x1 in [(xmin, x_split_l), (x_split_r, xmax)]:
        ax.add_line(Line2D([x0, x1], [ymax, ymax], **kw))  # top
        ax.add_line(Line2D([x0, x1], [ymin, ymin], **kw))  # bottom
        ax.add_line(Line2D([x0, x0], [ymin, ymax], **kw))  # left
        ax.add_line(Line2D([x1, x1], [ymin, ymax], **kw))  # right


with plt.rc_context(
    {
        "figure.figsize": (3, 3),
        "figure.dpi": 300,
        "font.size": 6,
        "lines.markersize": 1,
    }
):
    size = 3
    fig, (ax1, ax2) = plt.subplots(
        nrows=1,
        ncols=2,
        figsize=(18 / 2.54, 3),
        # sharey=True,
        width_ratios=(N_PROBLEMS + 1, N_OPTIMIZERS),
        layout="constrained",
    )

    # Successful starts over problems
    ax = ax1
    order = (
        df.groupby(["problem", "difficulty"])["n_success"]
        .median()
        .sort_values(ascending=False)
        .reset_index()
        .set_index("problem")
    )
    # insert placeholder between easy and hard
    order = (
        order.query("difficulty == 'easy'").index.tolist()
        + [""]
        + order.query("difficulty != 'easy'").index.tolist()
    )
    gap_idx = order.index("")

    sns.stripplot(
        data=df,
        x="problem",
        y="n_success",
        ax=ax,
        order=order,
        size=size,
    )
    sns.pointplot(
        data=df,
        x="problem",
        y="n_success",
        ax=ax,
        order=order,
        estimator="median",
        errorbar=None,
        markers="_",
        markeredgewidth=1,
        color="black",
        linestyle="none",
        zorder=10,
    )
    ax.set_yscale("symlog", linthresh=1)
    # set ylim according to data
    ax.set_ylim(-0.5, 10 ** np.ceil(np.log10(df.n_success.max())))
    ax.set_xlabel("Problem")
    ax.set_ylabel("# Successful starts")
    ax.tick_params("x", labelrotation=90)

    # hide tick at placeholder x
    tick = ax.xaxis.get_major_ticks()[gap_idx]
    tick.tick1line.set_visible(False)
    tick.tick2line.set_visible(False)
    add_split_spines(ax, gap_idx, xmax=len(order) - 0.5)

    # Successful starts over optimizers
    ax = ax2
    order = (
        df.groupby("optimizer_label")["n_success"]
        .median()
        .sort_values(ascending=False)
        .index
    )
    sns.stripplot(
        data=df,
        x="optimizer_label",
        y="n_success",
        ax=ax,
        order=order,
        size=size,
    )
    sns.pointplot(
        data=df,
        x="optimizer_label",
        y="n_success",
        ax=ax,
        order=order,
        estimator="median",
        errorbar=None,
        markers="_",
        markeredgewidth=1,
        color="black",
        linestyle="none",
        zorder=10,
    )
    ax.set_yscale("symlog", linthresh=1)
    # set ylim according to data
    ax.set_ylim(-0.5, 10 ** np.ceil(np.log10(df.n_success.max())))
    ax.set_xlabel("Optimisation method")
    ax.set_ylabel("# Successful starts")
    ax.tick_params("x", labelrotation=90)
    fig.align_xlabels()
    plt.savefig("out/Figure6DE.svg")
    plt.show()